In [4]:
import sys
from pathlib import Path
import torch
from omegaconf import OmegaConf

# -------------------------
# Add the folder containing your modules
# -------------------------
module_path = Path(r"C:\Users\tom39\Desktop\MHNfs\github").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# -------------------------
# Import your modules (same folder)
# -------------------------
from cross_attention_module import CrossAttentionModule
from context_module import ContextModule
from similarity_module import SimilarityModule
from mhnfs_final_model import MHNfsFinalModel

# -------------------------
# Dummy config
# -------------------------
cfg = OmegaConf.create({
    "model": {
        "transformer": {
            "activity_embedding_dim": 1024
        },
        "associationSpace_dim": 1024,
        "hopfield": {
            "num_steps": 2,
            "heads": 4,
            "beta": 1.0,
            "dropout": 0.1
        },
        "similarityModule": {
            "l2Norm": True,
            "numHeads": 4,
            "temperature": 1.0,
            "aggregation": "sum",
            "posWeight": 1.0,
            "negWeight": 1.0,
            "topk": None,
            "scaling": "1/N"
        }
    }
})

# -------------------------
# Instantiate modules
# -------------------------
cross_attention = CrossAttentionModule(cfg)
context_module = ContextModule(cfg)
similarity_module = SimilarityModule(
    cfg,
    input_dim=cfg.model.transformer.activity_embedding_dim
)

model = MHNfsFinalModel(
    cross_attention,
    context_module,
    similarity_module
)

model.eval()

# -------------------------
# Dummy input
# -------------------------
B = 4
D = 1024
Na = 5
Ni = 7
Nc = 20

query = torch.randn(B, 1, D)
actives = torch.randn(B, Na, D)
inactives = torch.randn(B, Ni, D)

mask_a = torch.ones(B, Na, dtype=torch.bool)
mask_i = torch.ones(B, Ni, dtype=torch.bool)

context = torch.randn(B, Nc, D)

# -------------------------
# Run forward pass
# -------------------------
with torch.no_grad():
    output = model(
        query,
        actives,
        inactives,
        mask_a,
        mask_i,
        context
    )

print("Output shape:", output.shape)
print("Output:", output)

Output shape: torch.Size([4, 1])
Output: tensor([[4.5267e-05],
        [3.7694e-05],
        [7.0238e-06],
        [7.6021e-05]])
